# Jumper su Colab — riprende C (`jumper.stairs_free`) con le correzioni dell'atterraggio

**Prima di tutto:** menu *Runtime → Cambia tipo di runtime → **T4 GPU** → Salva*.
Su Google Drive, cartella **`jumper`** (in *Il mio Drive*): `jumper-stairs-task.zip` (il codice della task) e `model_6750.pt` (il checkpoint di C).

**Come si usa:** *Runtime → Esegui tutto*. Alla prima cella Colab chiede l'accesso a Drive: *Connetti a Google Drive* → il tuo account → *Consenti*. Poi lascia la scheda aperta.

**Cosa fa:** clone del repository al commit bloccato, venv, torch e versioni identiche al backup di C, codice della task dallo zip su Drive → riparte dall'ultimo checkpoint salvato su Drive (`jumper/ckpt/`), altrimenti da `model_6750.pt` → mini-run di prova → training, fermato a orologio dopo `TRAIN_HOURS`. **Ogni `DRIVE_SYNC_MIN` minuti l'ultimo checkpoint completo va su Drive**: se Colab chiude la sessione (succede, sul piano gratuito), basta *Esegui tutto* di nuovo e riparte da lì. Alla fine: curve, metriche, replay MP4 e backup ZIP in `jumper/out/` su Drive.

**Cosa guardare nella cella 4 (una riga al minuto):** `level` (0–9 → alzata 2–15 cm), `steps_by_flight` (gradini saliti saltando per episodio: deve superare 1, cioè concatenare), `progress_m`. In TensorBoard/curve: `landing_dx_m` (distanza dal centro della pedata all'atterraggio, deve scendere verso 0.03–0.05) ed `edge_s`.


In [ ]:
# @title 1. Impostazioni (modifica solo questa cella)
import json
import math
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
import uuid
import venv
import zipfile
from collections import deque
from datetime import datetime, timezone
from pathlib import Path

# ── Cosa allenare ────────────────────────────────────────────────────────────
# A = "jumper.stairs" (cieco, esportabile); B = "jumper.stairs_sighted" (vede il terreno, deve saltare);
# C = "jumper.stairs_free" (vede il terreno, sale come vuole). Sessione 2: B contro C.
TASK = "jumper.stairs_free"            # C: vede il terreno, sale come vuole
SECOND_TASK = ""                       # Colab gratis ha una sola GPU
NUM_ENVS = 2048                        # per processo; su T4 16 GB sotto i 12 000 MiB di picco c'è margine
WORKFLOW_MODE = "Continue training"    # riparte da C (o dall'ultimo checkpoint su Drive)

# ── Tempo ────────────────────────────────────────────────────────────────────
TRAIN_HOURS = 10.0      # training; Colab gratis di solito chiude prima: i checkpoint su Drive coprono
SMOKE_ITERATIONS = 5    # mini-run di prova prima di quello lungo (0 per saltarlo)

# ── Warm start (New training, e le task in RESTART_TASKS) ────────────────────
WARM_START = True
WARM_SOURCE = "tasks/jumper/ref_free_jump/out/example/model_5999.pt"
# Task da far ripartire dal warm start anche in Continue (es. una policy che ha
# imparato una scappatoia del reward e non vale la pena continuare).
RESTART_TASKS = []
# Continue con codice della task cambiato: il manifest del backup viene riscritto
# con la configurazione attuale (osservazioni e rete devono restare uguali).
ACCEPT_CODE_CHANGES = True

# ── Sorgenti ─────────────────────────────────────────────────────────────────
REPOSITORY_URL = "https://github.com/KingKongRobotics/jumper.git"
SOURCE_REVISION = "b2e58669848a3ba4f209597cd6bebdfe9d0533a4"   # bloccato: la ripresa lo confronta
DRIVE = Path("/content/drive/MyDrive/jumper")   # la cartella su Google Drive
PATCH_DIR = str(DRIVE)                            # contiene jumper-stairs-task.zip
START_CHECKPOINT = DRIVE / "model_6750.pt"        # C, Kaggle 2026-10-27 (sha256 sotto)
START_SHA256 = "76e4afa0152755394674c28751794aa5f190aeda8b37521db8a6de01371ae5bc"
DRIVE_CKPT = DRIVE / "ckpt"                       # l'ultimo checkpoint completo, aggiornato durante il training
DRIVE_SYNC_MIN = 20
# Le versioni dei pacchetti del backup di C (run/colab-run.json, runtime_versions).
PINNED_VERSIONS = {"mujoco": "3.11.0", "mujoco-warp": "3.11.0", "numpy": "2.5.2",
                   "tensordict": "0.15.3", "torch": "2.14.1", "warp-lang": "1.18.0"}

# ── Replay ───────────────────────────────────────────────────────────────────
REPLAY_SECONDS = 20.0   # replay finale, per task
REPLAY_ROW = ""         # "" = la riga media raggiunta dal curriculum; oppure 0..9
REPLAY_WIDTH, REPLAY_HEIGHT, REPLAY_FPS = 960, 540, 30.0
# Camera: azimut 90 = di lato (0 = da dietro, 180 = di fronte), elevazione negativa = guarda in giù
REPLAY_AZIMUTH, REPLAY_ELEVATION, REPLAY_DISTANCE = 90.0, -8.0, 1.3
# Replay automatici durante il training (sulla GPU della task, dall'ultimo checkpoint
# completo): il primo dopo REPLAY_FIRST_HOURS, poi ogni REPLAY_EVERY_HOURS. 0 = mai.
REPLAY_EVERY_HOURS = 2.0
REPLAY_FIRST_HOURS = 0.5
REPLAY_PERIODIC_SECONDS = 10.0
REPLAY_MIN_FREE_MIB = 2500   # saltato se la GPU ha meno memoria libera di così

# ── Derivati ─────────────────────────────────────────────────────────────────
from google.colab import drive as _drive
_drive.mount("/content/drive")
if not DRIVE.is_dir():
    raise FileNotFoundError(f"manca la cartella {DRIVE}: creala su Google Drive e mettici zip e checkpoint")
WORK = Path("/content/work")      # effimero: sparisce con la sessione
OUT = DRIVE / "out"               # persistente: su Drive
REPO = WORK / "jumper"
PYTHON = REPO / ".venv/bin/python"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
SESSION = OUT / f"session-{STAMP}"
BACKUPS = OUT / "backups"
TASKS = [t for t in (TASK, SECOND_TASK) if t]
for directory in (WORK, OUT, SESSION, BACKUPS):
    directory.mkdir(parents=True, exist_ok=True)
if WORKFLOW_MODE not in ("New training", "Continue training"):
    raise ValueError("WORKFLOW_MODE: New training oppure Continue training")
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"Jumper vuole Python 3.10-3.13, qui c'è {sys.version}")
for t in TASKS:
    if not re.fullmatch(r"jumper\.[a-z_]+", t):
        raise ValueError(f"task non valida: {t}")
print(f"{WORKFLOW_MODE}: {TASKS} x {NUM_ENVS} env, {TRAIN_HOURS} h di training. Output in {OUT}")


In [ ]:
# @title 2. Setup: GPU, clone, venv, torch, EGL, codice della task  (~6-10 min)
def run(command, *, cwd=None, check=True, env=None, timeout_s=None, tail_lines=80):
    """Esegue e mostra l'output in diretta. Se scade `timeout_s`, SIGINT a tutto
    il gruppo (training e helper salvano e chiudono) e si rientra con quel codice."""
    print("+", " ".join(str(p) for p in command), flush=True)
    command = [str(p) for p in command]
    tail = deque(maxlen=tail_lines)
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    timer = None
    if timeout_s:
        timer = threading.Timer(timeout_s, lambda: os.killpg(process.pid, signal.SIGINT))
        timer.daemon = True
        timer.start()
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line[-4096:])
        code = process.wait()
        if code and check:
            raise subprocess.CalledProcessError(code, command, output="".join(tail))
        return subprocess.CompletedProcess(command, code, stdout="".join(tail))
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
                process.wait(timeout=30)
            except (ProcessLookupError, subprocess.TimeoutExpired):
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        raise
    finally:
        if timer:
            timer.cancel()
        process.stdout.close()


# ── GPU ──
run(["nvidia-smi", "-L"])
rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total", "--format=csv,noheader"], text=True,
).strip().splitlines()
if not rows:
    raise RuntimeError("Nessuna GPU: Runtime → Cambia tipo di runtime → T4 GPU")
gpus = [[p.strip() for p in r.split(",")] for r in rows]
capability = float(gpus[0][2])
if capability < 7.5:
    raise RuntimeError(f"GPU {gpus[0][1]} (compute capability {capability}) non supportata: scegli T4")
GPU_NAME = gpus[0][1]
if SECOND_TASK and len(gpus) < 2:
    raise RuntimeError("SECOND_TASK vuole due GPU: scegli T4 x2 oppure metti SECOND_TASK = ''")
print("GPU:", [g[1] for g in gpus])

# ── Clone al commit bloccato ──
if not REPO.exists():
    run(["git", "clone", REPOSITORY_URL, REPO])
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()

# ── Il codice della task, dal Dataset, sopra il clone ──
patch = Path(PATCH_DIR)
zips = sorted(patch.glob("jumper-stairs-task*.zip"))
if not zips:
    raise FileNotFoundError(f"manca jumper-stairs-task.zip in {patch} (Google Drive)")
if zips:
    with zipfile.ZipFile(zips[-1]) as z:
        names = z.namelist()
        if any(n.startswith("/") or ".." in Path(n).parts for n in names):
            raise ValueError("zip con percorsi non sicuri")
        z.extractall(REPO)
    print("Applicato", zips[-1], f"({len(names)} file)")
else:
    copied = 0
    for src in patch.rglob("*"):
        if src.is_file():
            rel = src.relative_to(patch)
            # la cartella del dataset può avere un livello in più (nome del dataset)
            parts = rel.parts
            if parts and parts[0] not in ("tasks", "tools", "notebooks", "docs"):
                rel = Path(*parts[1:])
            dst = REPO / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            copied += 1
    print(f"Copiati {copied} file da {patch}")
for required in ("tasks/jumper/stairs/env_cfg.py", "tools/warm_start.py"):
    if not (REPO / required).is_file():
        raise FileNotFoundError(f"manca {required}: il Dataset non contiene il codice della task")

# ── Continue: quale backup per ogni task, e le sue versioni dei pacchetti ──
import glob

def backup_manifest(archive):
    with zipfile.ZipFile(archive) as z:
        info = z.getinfo("run/colab-run.json")
        if info.file_size > 1024**2:
            raise ValueError("manifest troppo grande")
        return json.loads(z.read(info))

BACKUP_FOR = {}
# Su Colab non c'è un backup ZIP da ripristinare: si riparte da un checkpoint
# (cella 3), con le versioni dei pacchetti del backup di C.
CORE_VERSIONS = dict(PINNED_VERSIONS)

# ── venv isolato ──
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(REPO / ".venv")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--upgrade", "pip", "-q"])
try:
    run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "ffmpeg"], check=False)
except Exception as exc:  # noqa: BLE001
    print("apt-get non riuscito (continuo):", exc)
TORCH_INDEX = "cu126" if capability < 8.0 else "cu128"
run([PYTHON, "-m", "pip", "install", "-q", "torch==2.9.1", "torchvision==0.24.1",
     "--index-url", f"https://download.pytorch.org/whl/{TORCH_INDEX}"])
run([PYTHON, "-m", "pip", "install", "-q", "-e", str(REPO)])
pins = [f"{k}=={v}" for k, v in sorted(CORE_VERSIONS.items())]
if pins:
    run([PYTHON, "-m", "pip", "install", "-q", *pins])   # le versioni esatte del backup, come il notebook ufficiale
run([PYTHON, "-m", "pip", "install", "-q", "matplotlib"])

# ── Rendering headless per il replay MP4 (il training non ne ha bisogno) ──
# Il notebook ufficiale esige l'EGL di NVIDIA; l'immagine Kaggle non lo espone
# (misurato: nessuna libEGL_nvidia nei percorsi standard). Si cerca ovunque e,
# se manca, si ripiega sull'EGL software di Mesa: basta per 12 s di video a un
# solo robot. Il replay è comunque in try/except: non blocca mai il backup.
import glob as _glob
os.environ.update(MUJOCO_GL="egl", PYTHONUNBUFFERED="1", MPLBACKEND="Agg", PYOPENGL_PLATFORM="egl")
_egl_hits = []
for pattern in ("/usr/lib64-nvidia/libEGL_nvidia.so.*", "/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.*",
                "/usr/lib/x86_64-linux-gnu/nvidia/current/libEGL_nvidia.so.*", "/usr/lib/nvidia/libEGL_nvidia.so.*",
                "/usr/local/nvidia/lib64/libEGL_nvidia.so.*", "/usr/local/cuda*/lib64/libEGL_nvidia.so.*"):
    _egl_hits += [h for h in _glob.glob(pattern) if Path(h).is_file()]
if not _egl_hits:
    try:
        _egl_hits = subprocess.run(["bash", "-lc", "find / -xdev -name 'libEGL_nvidia.so.*' 2>/dev/null | head -5"],
                                   capture_output=True, text=True, timeout=120).stdout.split()
    except Exception:  # noqa: BLE001
        _egl_hits = []
if _egl_hits:
    lib = Path(_egl_hits[0]).resolve()
    (WORK / "nvidia-egl.json").write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": str(lib)}}))
    os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(WORK / "nvidia-egl.json")
    os.environ["LD_LIBRARY_PATH"] = ":".join([str(lib.parent), *[e for e in os.environ.get("LD_LIBRARY_PATH", "").split(":") if e]])
    print("EGL NVIDIA:", lib)
else:
    print("EGL NVIDIA assente: il replay userà l'EGL software di Mesa (più lento, ma basta per un video breve)")
    try:
        run(["apt-get", "install", "-y", "-qq", "libegl-mesa0", "libgl1-mesa-dri", "libgles2"], check=False)
    except Exception as exc:  # noqa: BLE001
        print("apt-get mesa non riuscito (continuo):", exc)
    os.environ["EGL_PLATFORM"] = "surfaceless"

run([PYTHON, "tools/colab.py", "check", "--repo", REPO, "--out", SESSION / "runtime.json"], cwd=REPO)
runtime = json.loads((SESSION / "runtime.json").read_text())
for pkg, want in CORE_VERSIONS.items():
    if runtime["versions"].get(pkg) != want:
        raise RuntimeError(f"versione di {pkg}: il backup vuole {want}, installata {runtime['versions'].get(pkg)}")
listing = run([PYTHON, "scripts/train.py", "--list"], cwd=REPO).stdout
for t in TASKS:
    if t not in listing:
        raise RuntimeError(f"{t} non compare in --list: manca la riga di import in tasks/__init__.py del Dataset")
run([PYTHON, "scripts/train.py", "--task", TASK, "--dry-run"], cwd=REPO)
print("\nSETUP OK —", GPU_NAME, "| commit", COMMIT[:9], "| versioni", runtime["versions"])


In [ ]:
# @title 3. Punto di partenza: ripristino del backup (Continue) oppure warm start (New)
START_FROM = {}   # task -> checkpoint da cui `tools/colab.py train --resume` riparte

# `tools/colab.py train --resume` vuole accanto al checkpoint il suo manifest con
# la configurazione e lo sha del file; questo helper lo scrive con le funzioni di
# colab.py stesso, così i campi di compatibilità sono esattamente quelli. Serve
# al warm start (che non ha manifest) e, con ACCEPT_CODE_CHANGES, a un backup
# fatto con una versione precedente del codice della task.
manifest_helper = r"""
import importlib.util, json, sys, uuid
from pathlib import Path
repo, task, model, num_envs, ckpt = Path(sys.argv[1]), sys.argv[2], sys.argv[3], int(sys.argv[4]), Path(sys.argv[5])
spec = importlib.util.spec_from_file_location("colab", repo / "tools/colab.py")
colab = importlib.util.module_from_spec(spec); spec.loader.exec_module(colab)
cfg = colab.make_config(repo, task=task, model=model, num_envs=num_envs)
old = {}
if (ckpt.parent / colab.MANIFEST).is_file():
    old = json.loads((ckpt.parent / colab.MANIFEST).read_text())
    changed = [k for k in colab.COMPATIBILITY if old.get("config", {}).get(k) != cfg[k]]
    print("campi diversi dal backup:", changed or "nessuno")
    if any(k not in ("source_sha256", "runtime_versions") for k in changed):
        raise SystemExit("il backup differisce in " + str(changed) + ": non e' una semplice modifica del codice")
manifest = {"schema": colab.SCHEMA, "operation": "warm-" + uuid.uuid4().hex, "config": cfg,
            "command": old.get("command", ["tools/warm_start.py"]), "status": "finished",
            "checkpoints": {ckpt.name: colab.sha256(ckpt)}, "checkpoint": str(ckpt),
            "resume_from": old.get("checkpoint"), "run_directory": str(ckpt.parent),
            "previous_manifest": old.get("operation")}
colab.atomic_json(ckpt.parent / colab.MANIFEST, manifest)
colab.validate_resume(ckpt, cfg)
print("manifest ok:", ckpt.parent / colab.MANIFEST)
"""


def warm_start(t):
    out = REPO / "logs" / "warm" / t / "model_0.pt"
    run([PYTHON, "tools/warm_start.py", "--task", t, "--from", WARM_SOURCE, "--out", out,
         "--num-envs", "4", "--device", "cpu"], cwd=REPO)
    run([PYTHON, "-c", manifest_helper, REPO, t, "jumper", str(NUM_ENVS), out], cwd=REPO)
    return str(out)


def drive_latest(t):
    """L'ultimo checkpoint che la cella 4 ha copiato su Drive per questa task, se c'è."""
    folder = DRIVE_CKPT / t
    found = sorted(folder.glob("model_*.pt"), key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
    return found[-1] if found else None


if WORKFLOW_MODE == "Continue training":
    import hashlib
    for t in TASKS:
        if t in RESTART_TASKS:
            print(t, ": riparte dal warm start (RESTART_TASKS)")
            START_FROM[t] = warm_start(t)
            continue
        source = drive_latest(t)
        if source is None:
            source = START_CHECKPOINT
            if not source.is_file():
                raise FileNotFoundError(f"manca {source} su Google Drive")
            digest = hashlib.sha256(source.read_bytes()).hexdigest()
            if digest != START_SHA256:
                raise ValueError(f"{source.name}: sha256 {digest[:12]}..., atteso {START_SHA256[:12]}...")
        dest = WORK / f"restored-{t}-{uuid.uuid4().hex[:8]}"
        dest.mkdir(parents=True)
        ckpt = dest / source.name
        shutil.copy2(source, ckpt)
        if (source.parent / "colab-run.json").is_file() and source.parent != DRIVE:
            shutil.copy2(source.parent / "colab-run.json", dest / "colab-run.json")
        # Il manifest per `colab.py train --resume`, scritto con la configurazione attuale
        # (il codice della task è cambiato: osservazioni e rete restano uguali).
        run([PYTHON, "-c", manifest_helper, REPO, t, "jumper", str(NUM_ENVS), ckpt], cwd=REPO)
        START_FROM[t] = str(ckpt)
        print(t, "riparte da", source)
elif WARM_START:
    for t in TASKS:
        START_FROM[t] = warm_start(t)
else:
    print("New training da zero (WARM_START = False)")
print("Partenze:", START_FROM or "da zero")


In [ ]:
# @title 4. Mini-run di prova, poi il training lungo (una task per GPU, fermato a orologio)
def train_env(gpu_index):
    """Ogni processo vede una sola GPU, come `cuda:0`: colab.py allena sempre su cuda:0."""
    e = os.environ.copy()
    e["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
    return e


def train_command(task, iterations, manifest, num_envs, resume=None):
    cmd = [str(PYTHON), "tools/colab.py", "train", "--repo", str(REPO), "--task", task, "--model", "jumper",
           "--num-envs", str(num_envs), "--iterations", str(iterations), "--manifest", str(manifest)]
    if resume:
        cmd += ["--resume", str(resume)]
    return cmd


# ── Mini-run: se la scena non si costruisce (nconmax, un termine che non si risolve,
# un checkpoint che non si carica) deve fallire qui, in 3 minuti, non dopo 10 ore. ──
if SMOKE_ITERATIONS > 0:
    # Tutte le task, una per GPU in parallelo: una scena che non si costruisce
    # deve fermare qui, non far perdere 10 h all'altra task.
    # Senza --resume: il manifest del warm start è scritto per NUM_ENVS e colab.py
    # rifiuta un altro conteggio. Il caricamento del checkpoint si vede nei primi
    # minuti del run lungo, nel suo log.
    smoke_procs = {}
    for gpu, t in enumerate(TASKS):
        smoke_manifest = SESSION / f"smoke-{t}.json"
        log = (SESSION / f"smoke-{t}.log").open("w")
        smoke_procs[t] = (subprocess.Popen(train_command(t, SMOKE_ITERATIONS, smoke_manifest, min(NUM_ENVS, 256)),
                                           cwd=REPO, env=train_env(gpu), stdout=log, stderr=subprocess.STDOUT,
                                           text=True, bufsize=1, start_new_session=True), log, smoke_manifest)
    for t, (p, log, smoke_manifest) in smoke_procs.items():
        try:
            p.wait(timeout=900)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        log.close()
        smoke = json.loads(smoke_manifest.read_text()) if smoke_manifest.is_file() else {"status": "missing"}
        if smoke.get("status") != "finished":
            print((SESSION / f"smoke-{t}.log").read_text(errors="replace")[-3000:])
            raise RuntimeError(f"il mini-run di {t} non è finito: {smoke.get('status')} {smoke.get('error')}")
        print(f"Mini-run OK ({t}):", smoke["run_directory"])

# ── Replay periodici: 10 s di video dall'ultimo checkpoint completo, senza fermare il training ──
RATES = {}
for t in TASKS:
    rf = SESSION / f"rates-{t}.json"
    try:
        run([PYTHON, "tools/colab.py", "rates", "--repo", REPO, "--task", t, "--model", "jumper", "--out", rf], cwd=REPO)
        RATES[t] = json.loads(rf.read_text())
    except Exception as exc:  # noqa: BLE001 -- senza le frequenze niente replay, ma il training parte lo stesso
        print(f"[replay {t}] frequenze non lette ({exc}): replay disattivati per questa task")
REPLAYS = SESSION / "replays"
REPLAYS.mkdir(exist_ok=True)
replay_locks = {t: threading.Lock() for t in TASKS}


def gpu_free_mib(gpu):
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits",
                                       "-i", str(gpu)], text=True)
        return float(out.strip().splitlines()[0])
    except Exception:  # noqa: BLE001
        return 0.0


def latest_complete_checkpoint(manifest_path):
    """L'ultimo model_N.pt che colab.py ha già registrato (quindi completo, con sha)."""
    m = json.loads(manifest_path.read_text())
    run_dir, recorded = m.get("run_directory"), m.get("checkpoints") or {}
    if not run_dir or not recorded:
        return None
    name = max(recorded, key=lambda n: int(re.search(r"model_(\d+)", n).group(1)))
    return Path(run_dir) / name


def replay_command(task, checkpoint, video, seconds, row, extra=()):
    steps = math.ceil(seconds * RATES[task]["control_hz"])
    cmd = [PYTHON, "scripts/play.py", "--task", task, "--model", "jumper", "--checkpoint", checkpoint,
           "--backend", "warp", "--device", "cuda:0", "--num_envs", "1", "--headless", "--steps", steps,
           "--physics-hz", RATES[task]["physics_hz"], "--video", video, "--video-fps", REPLAY_FPS,
           "--video-width", REPLAY_WIDTH, "--video-height", REPLAY_HEIGHT, "--strip-visual", "off",
           "--video-azimuth", REPLAY_AZIMUTH, "--video-elevation", REPLAY_ELEVATION,
           "--stairs-row", row, *extra]
    if REPLAY_DISTANCE:
        cmd += ["--video-distance", REPLAY_DISTANCE]
    return [str(c) for c in cmd]


def periodic_replay(task, gpu, row):
    if not replay_locks[task].acquire(blocking=False):
        return
    try:
        if task not in RATES:
            return
        ckpt = latest_complete_checkpoint(MANIFESTS[task])
        if ckpt is None or not ckpt.is_file():
            print(f"[replay {task}] nessun checkpoint completo ancora", flush=True)
            return
        free = gpu_free_mib(gpu)
        if free < REPLAY_MIN_FREE_MIB:
            print(f"[replay {task}] saltato: {free:.0f} MiB liberi sulla GPU {gpu}", flush=True)
            return
        it = re.search(r"model_(\d+)", ckpt.name).group(1)
        video = REPLAYS / f"{task}-it{it}-row{row}.mp4"
        r = subprocess.run(replay_command(task, ckpt, video, REPLAY_PERIODIC_SECONDS, row), cwd=REPO,
                           env=train_env(gpu), capture_output=True, text=True, timeout=1500)
        if r.returncode == 0 and video.is_file():
            print(f"[replay {task}] {video.relative_to(OUT)} ({video.stat().st_size / 1e6:.1f} MB)", flush=True)
        else:
            print(f"[replay {task}] fallito ({r.returncode}):\n{r.stdout[-1500:]}", flush=True)
    except Exception as exc:  # noqa: BLE001 -- un replay non ferma mai il training
        print(f"[replay {task}] errore: {exc}", flush=True)
    finally:
        replay_locks[task].release()


def drive_sync(task):
    """Copia su Drive l'ultimo checkpoint completo e il suo manifest; tiene solo l'ultimo."""
    try:
        ckpt = latest_complete_checkpoint(MANIFESTS[task])
        if ckpt is None or not ckpt.is_file():
            return
        folder = DRIVE_CKPT / task
        folder.mkdir(parents=True, exist_ok=True)
        if (folder / ckpt.name).is_file():
            return
        tmp = folder / (ckpt.name + ".part")
        shutil.copy2(ckpt, tmp)
        tmp.rename(folder / ckpt.name)
        shutil.copy2(MANIFESTS[task], folder / "colab-run.json")
        for old in folder.glob("model_*.pt"):
            if old.name != ckpt.name:
                old.unlink()
        print(f"[drive] {task}: {ckpt.name} salvato in {folder}", flush=True)
    except Exception as exc:  # noqa: BLE001 -- un salvataggio mancato non ferma il training
        print(f"[drive] {task}: salvataggio non riuscito ({exc})", flush=True)


# ── Il training lungo ──
MANIFESTS = {t: SESSION / f"training-{t}.json" for t in TASKS}
procs = {}
t0 = time.time()
next_replay = t0 + REPLAY_FIRST_HOURS * 3600 if REPLAY_EVERY_HOURS > 0 else None
next_sync = t0 + DRIVE_SYNC_MIN * 60
for gpu, t in enumerate(TASKS):
    cmd = train_command(t, 1_000_000, MANIFESTS[t], NUM_ENVS, START_FROM.get(t))
    print("+", " ".join(cmd), f"(GPU {gpu})", flush=True)
    log = (SESSION / f"train-{t}.log").open("w")
    procs[t] = (subprocess.Popen(cmd, cwd=REPO, env=train_env(gpu), stdout=log, stderr=subprocess.STDOUT,
                                 text=True, bufsize=1, start_new_session=True), log)

deadline = t0 + TRAIN_HOURS * 3600
stopped = False
PROGRESS = re.compile(r"Learning iteration (\d+)/|Mean reward:\s*([-0-9.]+)|Curriculum/stairs/level:\s*([-0-9.]+)|steps_by_flight:\s*([-0-9.]+)|progress_m:\s*([-0-9.]+)|steps_crawled:\s*([-0-9.]+)")
try:
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(60)
        now = time.time()
        status = []
        for t, (p, log) in procs.items():
            text = (SESSION / f"train-{t}.log").read_text(errors="replace")[-20000:]
            it = PROGRESS.findall(text)
            last_it = [m[0] for m in it if m[0]]
            last_rew = [m[1] for m in it if m[1]]
            last_lvl = [m[2] for m in it if m[2]]
            last_jmp = [m[3] for m in it if m[3]]
            last_prog = [m[4] for m in it if m[4]]
            last_crawl = [m[5] for m in it if m[5]]
            status.append(f"{t}: it {last_it[-1] if last_it else '-'} reward {last_rew[-1] if last_rew else '-'}"
                          f" level {last_lvl[-1] if last_lvl else '-'} steps_by_flight {last_jmp[-1] if last_jmp else '-'}"
                          f" crawled {last_crawl[-1] if last_crawl else '-'} progress_m {last_prog[-1] if last_prog else '-'}"
                          f"{'' if p.poll() is None else f' [uscito {p.returncode}]'}")
        print(f"[{(now - t0) / 3600:5.2f} h] " + " | ".join(status), flush=True)
        if next_replay and now >= next_replay and not stopped:
            next_replay = now + REPLAY_EVERY_HOURS * 3600
            for gpu, t in enumerate(TASKS):
                if procs[t][0].poll() is None:
                    text = (SESSION / f"train-{t}.log").read_text(errors="replace")[-20000:]
                    lv = [m[2] for m in PROGRESS.findall(text) if m[2]]
                    row = max(0, min(9, int(round(float(lv[-1]))))) if lv else 0
                    threading.Thread(target=periodic_replay, args=(t, gpu, row), daemon=True).start()
        if now >= next_sync:
            next_sync = now + DRIVE_SYNC_MIN * 60
            for t in TASKS:
                drive_sync(t)
        if now >= deadline and not stopped:
            print(">>> budget di tempo esaurito: SIGINT (il checkpoint più recente è già su disco)", flush=True)
            for p, _ in procs.values():
                if p.poll() is None:
                    os.killpg(p.pid, signal.SIGINT)
            stopped = True
        if stopped and now >= deadline + 300:
            for p, _ in procs.values():
                if p.poll() is None:
                    os.killpg(p.pid, signal.SIGKILL)
finally:
    for p, log in procs.values():
        try:
            p.wait(timeout=120)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        log.close()

for t in TASKS:
    drive_sync(t)

TRAINING = {}
RUN_DIR = {}
CHECKPOINT = {}
for t in list(TASKS):
    m = json.loads(MANIFESTS[t].read_text())
    print(f"{t}: stato {m['status']}, run {m.get('run_directory')}, checkpoint {m.get('checkpoint')}")
    if m["status"] not in ("finished", "interrupted") or not m.get("checkpoint"):
        # L'altra task ha comunque le sue 10 h: si va avanti senza questa.
        print((SESSION / f"train-{t}.log").read_text(errors="replace")[-3000:])
        print(f"!!! {t}: training {m['status']}: {m.get('error')} -- esclusa da curve, replay e backup")
        TASKS.remove(t)
        continue
    TRAINING[t] = m
    RUN_DIR[t] = Path(m["run_directory"])
    CHECKPOINT[t] = Path(m["checkpoint"])
if not TASKS:
    raise RuntimeError("nessuna task ha prodotto un checkpoint")
print(f"Training chiuso dopo {(time.time() - t0) / 3600:.2f} h")


In [ ]:
# @title 5. Curve, metriche della scalinata, replay MP4, export ONNX, backup ZIP
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, Video, display

ARTIFACTS = {}
KEY_TAGS = (
    "Metrics/stairs/steps_by_flight", "Metrics/stairs/steps_crawled", "Metrics/stairs/crawl_ratio",
    "Metrics/stairs/jumps", "Metrics/stairs/progress_m", "Metrics/stairs/apex_rise", "Metrics/stairs/flight_s", "Metrics/stairs/max_step",
    "Metrics/stairs/reached_top", "Curriculum/stairs/level", "Curriculum/stairs/level_max",
    "Curriculum/stairs/riser_m", "Curriculum/stairs/promoted", "Curriculum/stairs/recent",
    "Episode_Termination/fell_over", "Episode_Termination/fell_off", "Episode_Termination/off_side",
    "Episode_Reward/illegal_contact", "Episode_Reward/crawl", "Episode_Reward/climbed_stream",
    "Train/mean_reward", "Policy/mean_noise_std",
)


def finite(points):
    return [(p["step"], p["value"]) for p in points if isinstance(p["value"], float)]


for t in TASKS:
    art = SESSION / "artifacts" / t
    art.mkdir(parents=True, exist_ok=True)
    ARTIFACTS[t] = art
    for name in ("runtime.json",):
        shutil.copy2(SESSION / name, art / name)

    # ── Curve: tutte le scalari (JSON+CSV+PNG ufficiali) e un pannello della scalinata ──
    metrics_json = art / "metrics.json"
    run([PYTHON, "tools/colab.py", "metrics", "--run", RUN_DIR[t], "--out", metrics_json,
         "--plot", art / "training-curves.png"], cwd=REPO)
    tags = json.loads(metrics_json.read_text())["tags"]
    present = [k for k in KEY_TAGS if k in tags and finite(tags[k])]
    if present:
        cols = 3
        rows = math.ceil(len(present) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 2.8 * rows), squeeze=False)
        for ax, key in zip(axes.flat, present, strict=False):
            xs, ys = zip(*finite(tags[key]), strict=False)
            ax.plot(xs, ys, lw=1)
            ax.set_title(key, fontsize=9)
            ax.grid(alpha=0.3)
        for ax in list(axes.flat)[len(present):]:
            ax.axis("off")
        fig.suptitle(f"{t} — {RUN_DIR[t].name}")
        fig.tight_layout()
        fig.savefig(art / "stairs-curves.png", dpi=110)
        plt.close(fig)
        display(Image(filename=str(art / "stairs-curves.png")))
    summary = {}
    for key in KEY_TAGS:
        pts = finite(tags.get(key, []))
        if pts:
            last = pts[-1]
            tail = [v for _, v in pts[-20:]]
            summary[key] = {"last_step": last[0], "last": last[1], "mean_last_20": sum(tail) / len(tail)}
    (art / "stairs-summary.json").write_text(json.dumps(summary, indent=2))
    print(f"\n=== {t}: ultime 20 iterazioni (media) ===")
    for key, v in summary.items():
        print(f"  {key:<42} {v['mean_last_20']:>10.4f}   (ultimo {v['last']:.4f} @ it {v['last_step']})")

    # ── Replay MP4: la riga più alta raggiunta dal curriculum, dal pianerottolo ──
    level = summary.get("Curriculum/stairs/level_max", summary.get("Curriculum/stairs/level", {})).get("mean_last_20", 0.0)
    row = REPLAY_ROW if REPLAY_ROW != "" else str(max(0, min(9, int(round(level)))))
    # i replay periodici del training finiscono nel backup insieme a tutto il resto
    if REPLAYS.is_dir() and any(REPLAYS.iterdir()):
        shutil.copytree(REPLAYS, art / "replays", dirs_exist_ok=True)
    video = art / f"replay-{t}-row{row}.mp4"
    try:
        run(replay_command(t, CHECKPOINT[t], video, REPLAY_SECONDS, row,
                           extra=("--measure", "--replay-info-out", art / "replay-info.json")),
            cwd=REPO, env=train_env(0))
        display(Video(str(video), embed=True))
        for name in ("measure.csv", "measure.png"):
            src = CHECKPOINT[t].parent / "measure" / name
            if src.is_file():
                shutil.copy2(src, art / name)
    except Exception as exc:  # noqa: BLE001 -- il replay non deve impedire il backup
        print(f"[replay {t}] fallito: {exc}")
        video = None

    # ── Export ONNX: solo la task cieca può passare il controllo di esportabilità ──
    export_dir = None
    if t == "jumper.stairs":
        export_dir = SESSION / f"onnx-{t}"
        try:
            run([PYTHON, "scripts/export.py", "--task", t, "--model", "jumper", "--checkpoint", CHECKPOINT[t],
                 "--backend", "warp", "--device", "cuda:0", "--num_envs", "1", "--no-video", "--out", export_dir],
                cwd=REPO, env=train_env(0))
        except Exception as exc:  # noqa: BLE001
            print(f"[export {t}] fallito: {exc}")
            export_dir = None
    else:
        print(f"[export {t}] saltato: questa task non è esportabile per costruzione")

    # ── Backup ZIP: checkpoint + manifest + curve + replay + export, in /kaggle/working ──
    backup = BACKUPS / f"{t}-{STAMP}.zip"
    cmd = [PYTHON, "tools/colab.py", "backup", "--run", RUN_DIR[t], "--out", backup, "--extras", art]
    if export_dir:
        cmd += ["--export", export_dir]
    if video and video.is_file():
        cmd += ["--video", video]
    run(cmd, cwd=REPO)
    print(f"BACKUP {t}: {backup} ({backup.stat().st_size / 1e6:.0f} MB)")

print(f"\nFATTO. Tutto è su Google Drive in {OUT}; l'ultimo checkpoint in {DRIVE_CKPT}.")
print("Per continuare: Runtime → Esegui tutto (riparte dall'ultimo checkpoint su Drive).")


## Se qualcosa va storto

- **`nconmax overflow (nconmax must be >= N)`** nel mini-run: nello zip, in `tasks/jumper/stairs/env_cfg.py`, porta `cfg.sim.nconmax` esattamente a N e rilancia.
- **`cannot resume from ...`** all'inizio del run lungo: il warm start non combacia con la rete (osservazioni cambiate). Metti `WARM_START = False` per partire da zero, e segnalamelo.
- **`resume configuration differs`** in Continue: commit, `NUM_ENVS` o versioni dei pacchetti diversi dal backup. `SOURCE_REVISION` e `NUM_ENVS` devono essere quelli del backup; le versioni le reinstalla la cella 2.
- **Sessione interrotta da Colab** (inattività, limite d'uso della GPU gratuita): l'ultimo checkpoint è su Drive in `jumper/ckpt/` (al massimo `DRIVE_SYNC_MIN` minuti indietro). *Runtime → Esegui tutto* riparte da lì. Se Colab non dà la T4 ("non è possibile connettersi a un backend GPU"), riprova più tardi.
- **Il replay fallisce** ma il backup c'è: va bene, il video si rifà in un'altra sessione dal backup.
